Answer the problem in the context of topics covered in SDS. You are **not allowed** to seek or receive help from anyone to solve the problem. To keep the problem at reasonable difficulty, you are also **not allowed** to use LLMs such as ChatGPT, Gemini, Claude and Llama. You can **only use built-in libraries** of Apache Spark. For example, you cannot use Spark packages.

# Problem 4 [25 pts]

In this problem, you will be working with a network of power connections defined by `pc_nodes.csv` and `pc_edges.csv`, which is the same as in Problem 3.

### Added setup cell

This cell loads `pc_nodes.csv` and `pc_edges.csv`, then prepares reusable graph structures using only built-in Apache Spark functionality.

In [ ]:
# Spark setup and CSV loading
from pyspark.sql import SparkSession, functions as F, types as T, Window
from functools import reduce
import math

spark = SparkSession.builder.appName("SDS Power Connections").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

NODES_PATH = "pc_nodes.csv"
EDGES_PATH = "pc_edges.csv"

nodes_raw = spark.read.csv(NODES_PATH, header=True, inferSchema=True)
edges_raw = spark.read.csv(EDGES_PATH, header=True, inferSchema=True)

print("nodes columns:", nodes_raw.columns)
print("edges columns:", edges_raw.columns)

def first_existing(cols, candidates):
    lower_map = {c.lower(): c for c in cols}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    raise ValueError(f"None of {candidates} found in columns {cols}")

node_id_col = first_existing(nodes_raw.columns, ["id", "node_id", "Id", "ID"])
topic_col = first_existing(nodes_raw.columns, ["topic", "type", "label", "category"])
src_col = first_existing(edges_raw.columns, ["src", "source", "from", "source_id", "u"])
dst_col = first_existing(edges_raw.columns, ["dst", "target", "to", "target_id", "v"])

nodes = (
    nodes_raw
    .select(F.col(node_id_col).cast("string").alias("id"),
            F.col(topic_col).cast("string").alias("topic"))
    .dropDuplicates(["id"])
)

edges_directed = (
    edges_raw
    .select(F.col(src_col).cast("string").alias("src"),
            F.col(dst_col).cast("string").alias("dst"))
    .where(F.col("src").isNotNull() & F.col("dst").isNotNull() & (F.col("src") != F.col("dst")))
    .dropDuplicates()
)

# Power connections are physical connections, so treating them as undirected is usually appropriate.
# Set this to False if the CSV represents explicitly directed flow.
TREAT_AS_UNDIRECTED = True

if TREAT_AS_UNDIRECTED:
    edges = edges_directed.union(edges_directed.select(F.col("dst").alias("src"), F.col("src").alias("dst"))).dropDuplicates()
else:
    edges = edges_directed

nodes.cache(); edges.cache()
print("nodes:", nodes.count(), "edges used:", edges.count())
nodes.show(5, truncate=False)
edges.show(5, truncate=False)


## Shared helper functions for Problem 4

The code below builds an undirected adjacency list, computes connected components, and computes modularity. These helpers are used by both Girvan-Newman and SimRank-based partitioning.


In [ ]:
# Build undirected edges for community detection, regardless of PageRank/HITS direction settings.
undirected_edges = (
    edges_directed
    .select("src", "dst")
    .union(edges_directed.select(F.col("dst").alias("src"), F.col("src").alias("dst")))
    .where(F.col("src") != F.col("dst"))
    .dropDuplicates()
    .cache()
)

all_node_ids = [r["id"] for r in nodes.select("id").dropDuplicates().collect()]

adjacency = (
    undirected_edges
    .rdd
    .map(lambda r: (r["src"], r["dst"]))
    .groupByKey()
    .mapValues(lambda xs: sorted(set(xs)))
    .collectAsMap()
)

# Include isolated nodes.
adjacency = {node: list(adjacency.get(node, [])) for node in all_node_ids}

from collections import deque, defaultdict

def connected_components_from_adj(adj):
    visited = set()
    components = []
    for start in adj:
        if start in visited:
            continue
        q = deque([start])
        visited.add(start)
        comp = []
        while q:
            u = q.popleft()
            comp.append(u)
            for v in adj.get(u, []):
                if v not in visited:
                    visited.add(v)
                    q.append(v)
        components.append(sorted(comp))
    return components

def make_membership(components):
    mapping = {}
    for i, comp in enumerate(components):
        for node in comp:
            mapping[node] = i
    return mapping

def modularity(adj, components):
    # Undirected modularity: Q = sum_c [L_c/m - (d_c/(2m))^2]
    m = sum(len(nei) for nei in adj.values()) / 2.0
    if m == 0:
        return 0.0
    membership = make_membership(components)
    internal_edges_twice = defaultdict(int)
    degree_sum = defaultdict(int)
    for u, nbrs in adj.items():
        cu = membership[u]
        degree_sum[cu] += len(nbrs)
        for v in nbrs:
            if membership.get(v) == cu:
                internal_edges_twice[cu] += 1
    q = 0.0
    for c in range(len(components)):
        L_c = internal_edges_twice[c] / 2.0
        d_c = degree_sum[c]
        q += (L_c / m) - ((d_c / (2.0 * m)) ** 2)
    return q

def components_to_df(components):
    rows = [(i, node) for i, comp in enumerate(components) for node in comp]
    return spark.createDataFrame(rows, ["community", "id"])

initial_components = connected_components_from_adj(adjacency)
print("Initial connected components:", len(initial_components))
print("Initial modularity:", modularity(adjacency, initial_components))


## Problem 4a [10 pts]

Using Apache Spark, partition the network using the Girvan-Newman algorithm. Justify the choice of number of communities. Show the members of the largest community.

### Added solution for Problem 4a

## Problem 4a [10 pts] — Girvan-Newman partitioning

Girvan-Newman finds communities by repeatedly removing edges with high **edge betweenness**. An edge has high betweenness when many shortest paths pass through it. Removing such edges tends to separate communities.

The number of communities is chosen using **modularity**. We try multiple removals, compute modularity after each split, and choose the partition with the highest modularity.


In [ ]:
# Girvan-Newman edge betweenness using Brandes' algorithm for unweighted graphs

def single_source_edge_betweenness(root, adj):
    stack = []
    parents = defaultdict(list)
    sigma = defaultdict(float)
    dist = {}
    sigma[root] = 1.0
    dist[root] = 0
    q = deque([root])

    while q:
        v = q.popleft()
        stack.append(v)
        for w in adj.get(v, []):
            if w not in dist:
                dist[w] = dist[v] + 1
                q.append(w)
            if dist[w] == dist[v] + 1:
                sigma[w] += sigma[v]
                parents[w].append(v)

    delta = defaultdict(float)
    edge_credit = defaultdict(float)

    while stack:
        w = stack.pop()
        for v in parents[w]:
            credit = (sigma[v] / sigma[w]) * (1.0 + delta[w])
            edge = tuple(sorted((v, w)))
            edge_credit[edge] += credit
            delta[v] += credit

    return list(edge_credit.items())

def compute_edge_betweenness_spark(adj):
    bc_adj = spark.sparkContext.broadcast(adj)
    roots = list(adj.keys())
    raw = (
        spark.sparkContext
        .parallelize(roots, min(len(roots), spark.sparkContext.defaultParallelism * 4))
        .flatMap(lambda root: single_source_edge_betweenness(root, bc_adj.value))
        .reduceByKey(lambda a, b: a + b)
        .mapValues(lambda x: x / 2.0)  # undirected graph counts paths twice
    )
    result = raw.collect()
    bc_adj.unpersist()
    return dict(result)

def remove_edge_from_adj(adj, edge):
    u, v = edge
    new_adj = {x: list(nbrs) for x, nbrs in adj.items()}
    if v in new_adj.get(u, []):
        new_adj[u].remove(v)
    if u in new_adj.get(v, []):
        new_adj[v].remove(u)
    return new_adj

# To keep the exam implementation tractable, cap the number of edge-removal rounds.
# Increase MAX_REMOVALS if the graph is small and you want a more exhaustive search.
MAX_REMOVALS = 30

gn_history = []
current_adj = {u: list(vs) for u, vs in adjacency.items()}
best_components = connected_components_from_adj(current_adj)
best_q = modularity(adjacency, best_components)  # modularity is evaluated against original graph

for step in range(MAX_REMOVALS + 1):
    comps = connected_components_from_adj(current_adj)
    q = modularity(adjacency, comps)
    gn_history.append((step, len(comps), q, comps))
    print(f"step={step}, communities={len(comps)}, modularity={q:.6f}")

    if q > best_q:
        best_q = q
        best_components = comps

    if step == MAX_REMOVALS:
        break

    bet = compute_edge_betweenness_spark(current_adj)
    if not bet:
        break
    max_bet = max(bet.values())
    edges_to_remove = [e for e, b in bet.items() if abs(b - max_bet) < 1e-12]
    print("  removing", len(edges_to_remove), "edge(s) with betweenness", max_bet)
    for e in edges_to_remove:
        current_adj = remove_edge_from_adj(current_adj, e)

print("Best Girvan-Newman modularity:", best_q)
print("Chosen number of communities:", len(best_components))

gn_communities_df = components_to_df(best_components)

gn_largest_community_id = (
    gn_communities_df.groupBy("community").count().orderBy(F.desc("count"), F.asc("community")).first()["community"]
)

gn_largest_members = (
    gn_communities_df
    .filter(F.col("community") == gn_largest_community_id)
    .join(nodes, on="id", how="left")
    .orderBy("id")
)

print("Members of largest Girvan-Newman community")
gn_largest_members.show(200, truncate=False)


### Justification for number of Girvan-Newman communities

The chosen number of communities is the one with the highest modularity in the `gn_history` table. Modularity compares the density of edges inside communities against what would be expected by chance. A higher modularity means the partition has clearer community structure.


## Problem 4b [10 pts]

Using Apache Spark, partition the network using SimRank. Justify the choice of number of communities. Show the members of the largest community.

### Added solution for Problem 4b

## Problem 4b [10 pts] — SimRank-based partitioning

SimRank is a node-similarity method: two nodes are similar if they are connected to similar neighbors. Since SimRank produces similarity scores rather than communities directly, we convert it into communities by:

1. computing pairwise SimRank similarities,
2. linking pairs whose similarity is at least a threshold,
3. taking connected components in the similarity graph,
4. choosing the threshold that gives the highest modularity.


In [ ]:
# SimRank implementation for an undirected graph.
# Formula: s(a,b) = C / (|N(a)| |N(b)|) * sum_{i in N(a)} sum_{j in N(b)} s(i,j)

C = 0.8
SIMRANK_ITERS = 5

node_list = sorted(adjacency.keys())
node_set = set(node_list)

# Pair dictionary only stores unordered pairs (a,b), a <= b.
def pair_key(a, b):
    return (a, b) if a <= b else (b, a)

sim = {}
for a in node_list:
    for b in node_list:
        if a <= b:
            sim[(a, b)] = 1.0 if a == b else 0.0

def get_sim(sim_dict, a, b):
    return sim_dict[pair_key(a, b)]

for it in range(SIMRANK_ITERS):
    bc_sim = spark.sparkContext.broadcast(sim)
    bc_adj = spark.sparkContext.broadcast(adjacency)

    pairs = [(node_list[i], node_list[j]) for i in range(len(node_list)) for j in range(i, len(node_list))]

    def compute_pair(pair):
        a, b = pair
        if a == b:
            return (pair_key(a, b), 1.0)
        Na = bc_adj.value.get(a, [])
        Nb = bc_adj.value.get(b, [])
        if not Na or not Nb:
            return (pair_key(a, b), 0.0)
        total = 0.0
        old = bc_sim.value
        for i in Na:
            for j in Nb:
                total += old[pair_key(i, j)]
        return (pair_key(a, b), C * total / (len(Na) * len(Nb)))

    sim = dict(
        spark.sparkContext
        .parallelize(pairs, min(len(pairs), spark.sparkContext.defaultParallelism * 4))
        .map(compute_pair)
        .collect()
    )
    bc_sim.unpersist(); bc_adj.unpersist()
    print(f"finished SimRank iteration {it+1}")

# Candidate thresholds. If these produce either one giant community or all singletons,
# add more thresholds based on the observed similarity distribution.
positive_sims = sorted([v for (a, b), v in sim.items() if a != b and v > 0.0])
if positive_sims:
    quantile_thresholds = []
    for q in [0.50, 0.60, 0.70, 0.80, 0.90, 0.95]:
        idx = min(len(positive_sims) - 1, int(q * (len(positive_sims) - 1)))
        quantile_thresholds.append(positive_sims[idx])
    thresholds = sorted(set([0.05, 0.10, 0.15, 0.20, 0.25, 0.30] + quantile_thresholds))
else:
    thresholds = [1.0]

def sim_components_at_threshold(threshold):
    sim_adj = {u: [] for u in node_list}
    for (a, b), value in sim.items():
        if a != b and value >= threshold:
            sim_adj[a].append(b)
            sim_adj[b].append(a)
    return connected_components_from_adj(sim_adj)

simrank_history = []
best_sim_components = None
best_sim_q = -999.0
best_threshold = None

for threshold in thresholds:
    comps = sim_components_at_threshold(threshold)
    q = modularity(adjacency, comps)
    simrank_history.append((threshold, len(comps), q, comps))
    print(f"threshold={threshold:.6f}, communities={len(comps)}, modularity={q:.6f}")
    if q > best_sim_q:
        best_sim_q = q
        best_threshold = threshold
        best_sim_components = comps

print("Best SimRank threshold:", best_threshold)
print("Best SimRank modularity:", best_sim_q)
print("Chosen number of communities:", len(best_sim_components))

simrank_communities_df = components_to_df(best_sim_components)

simrank_largest_community_id = (
    simrank_communities_df.groupBy("community").count().orderBy(F.desc("count"), F.asc("community")).first()["community"]
)

simrank_largest_members = (
    simrank_communities_df
    .filter(F.col("community") == simrank_largest_community_id)
    .join(nodes, on="id", how="left")
    .orderBy("id")
)

print("Members of largest SimRank community")
simrank_largest_members.show(200, truncate=False)


### Justification for number of SimRank communities

SimRank itself produces similarities, not a fixed number of communities. Therefore, the notebook tests several similarity thresholds and chooses the threshold that maximizes modularity. The selected number of communities is the number of connected components in the thresholded SimRank similarity graph at the best threshold.


## Problem 4c [5 pts]

Using Apache Spark, compute the clustering coefficient of the network.

### Added solution for Problem 4c

## Problem 4c [5 pts] — Clustering coefficient of the network

The global clustering coefficient/transitivity is:

\[
C = \frac{3 	imes 	ext{number of triangles}}{	ext{number of connected triples}}
\]

The code below computes triangles and connected triples using Spark DataFrames.


In [ ]:
# ============================================================
# Triangle count and global clustering coefficient
# ============================================================
# This fixed version avoids ambiguous-column errors by:
# 1. canonicalizing undirected edges as (u, v), where u < v;
# 2. renaming intermediate columns before joining the third edge;
# 3. using aliases and qualified column references.

from pyspark.sql import functions as F

# Convert directed edges into undirected canonical edges.
# For an undirected edge between a and b:
# u = smaller endpoint
# v = larger endpoint
canon_edges = (
    edges
    .select(
        F.least(F.col("src"), F.col("dst")).alias("u"),
        F.greatest(F.col("src"), F.col("dst")).alias("v")
    )
    .where(F.col("u") != F.col("v"))
    .dropDuplicates(["u", "v"])
    .cache()
)

# Build node-neighbor relationships for degree / connected triples.
neighbors = (
    canon_edges.select(F.col("u").alias("id"), F.col("v").alias("neighbor"))
    .unionByName(canon_edges.select(F.col("v").alias("id"), F.col("u").alias("neighbor")))
    .dropDuplicates(["id", "neighbor"])
    .cache()
)

degree_df = (
    neighbors
    .groupBy("id")
    .agg(F.count("*").alias("degree"))
    .cache()
)

# Connected triples:
# For each node with degree d, number of unordered neighbor pairs is d choose 2.
connected_triples_df = (
    degree_df
    .withColumn(
        "connected_triples",
        F.when(
            F.col("degree") >= 2,
            (F.col("degree") * (F.col("degree") - F.lit(1))) / F.lit(2)
        ).otherwise(F.lit(0.0))
    )
)

connected_triples = (
    connected_triples_df
    .agg(F.sum("connected_triples").alias("connected_triples"))
    .first()["connected_triples"]
)

if connected_triples is None:
    connected_triples = 0.0

# Triangle counting:
# A triangle exists when we have edges:
# (u, v), (u, w), and (v, w), with u < v < w.
#
# Step 1: find pairs of edges sharing the same u:
# (u, v) and (u, w), with v < w.
e1 = canon_edges.alias("e1")
e2 = canon_edges.alias("e2")

open_wedges = (
    e1
    .join(e2, F.col("e1.u") == F.col("e2.u"), "inner")
    .where(F.col("e1.v") < F.col("e2.v"))
    .select(
        F.col("e1.u").alias("tri_u"),
        F.col("e1.v").alias("tri_v"),
        F.col("e2.v").alias("tri_w")
    )
    .cache()
)

# Step 2: check whether the closing edge (v, w) exists.
e3 = canon_edges.select(
    F.col("u").alias("edge_u"),
    F.col("v").alias("edge_v")
).alias("e3")

triangles_df = (
    open_wedges.alias("ow")
    .join(
        e3,
        (F.col("ow.tri_v") == F.col("e3.edge_u")) &
        (F.col("ow.tri_w") == F.col("e3.edge_v")),
        "inner"
    )
    .select(
        F.col("ow.tri_u").alias("u"),
        F.col("ow.tri_v").alias("v"),
        F.col("ow.tri_w").alias("w")
    )
    .dropDuplicates(["u", "v", "w"])
    .cache()
)

num_triangles = triangles_df.count()

# Global clustering coefficient / transitivity:
# Each triangle contributes 3 closed triples.
if connected_triples == 0:
    global_clustering_coefficient = 0.0
else:
    global_clustering_coefficient = (3.0 * num_triangles) / connected_triples

print("Number of undirected canonical edges:", canon_edges.count())
print("Number of triangles:", num_triangles)
print("Number of connected triples:", connected_triples)
print("Global clustering coefficient:", global_clustering_coefficient)

print("Sample triangles:")
triangles_df.show(20, truncate=False)

print("Node degrees and connected triples:")
connected_triples_df.orderBy(F.col("degree").desc()).show(20, truncate=False)

### Interpretation

A clustering coefficient near `0` means the network has little triangle closure: neighbors of a node are usually not connected to each other. A clustering coefficient closer to `1` means strong local clustering: if two nodes are both connected to the same node, they are also likely connected to each other.


## Fix summary

The previous error was:

```text
[AMBIGUOUS_REFERENCE] Reference `v` is ambiguous
```

This happened because both the intermediate triangle candidate table and `e3` had a column named `v` during the join.

The fixed version renames the triangle candidate columns to:

```python
tri_u, tri_v, tri_w
```

and renames the closing-edge columns to:

```python
edge_u, edge_v
```

Then the join condition uses fully qualified columns, avoiding ambiguity.